# NeuralEdge AI Boardroom — GRPO Training (Qwen3-0.6B)

**Theme 1 — Multi-Agent Interactions** • OpenEnv Hackathon submission

This notebook trains a Qwen3-0.6B agent (via TRL `GRPOTrainer` + Unsloth LoRA) to play CEO in the BoardSim environment. The agent must build winning coalitions among 4 hidden-agenda NPC board members across 10 rounds of market crises.

**Runs end-to-end on a free Colab T4** (~3-5 hours for 500 GRPO steps). Set `ENV_BASE_URL` below to your deployed HF Space or leave empty to launch a local Docker image.

## 1. Install dependencies

In [17]:
!pip install -q openenv-core==0.2.3 trl>=0.12 transformers>=4.45 datasets>=3.0 accelerate wandb matplotlib
!pip install -q --no-deps unsloth

## 2. Authenticate (HF + W&B)

The cell below will prompt you to securely enter your `HF_TOKEN` and `WANDB_API_KEY`.

In [18]:
import os

# ── Option 1 (Colab): add HF_TOKEN and WANDB_API_KEY to Colab Secrets (key icon in left sidebar)
# ── Option 2 (quick test): set them as env vars below before running
try:
    from google.colab import userdata
    os.environ.setdefault('HF_TOKEN',      userdata.get('HF_TOKEN') or '')
    os.environ.setdefault('WANDB_API_KEY', userdata.get('WANDB_API_KEY') or '')
except Exception:
    pass

# Fallback: set manually if Colab Secrets not configured
if not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = input('HF token not found in secrets. Paste it here: ').strip()
if not os.environ.get('WANDB_API_KEY'):
    os.environ['WANDB_API_KEY'] = input('WandB key not found in secrets. Paste it here: ').strip()

from huggingface_hub import login as hf_login
hf_login(token=os.environ['HF_TOKEN'])

import wandb
wandb.login(key=os.environ['WANDB_API_KEY'])
print('Authenticated successfully.')

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


Authenticated successfully.


## 3. Connect to the BoardSim environment

Set `ENV_BASE_URL` to your deployed HF Space URL (preferred). If empty, the cell will clone the repo and launch Docker locally — only works on a machine with Docker (NOT free Colab).

In [20]:
import os, sys

ENV_BASE_URL = 'https://stavankhobare-sst-metaxpytorch-hackathon.hf.space'
REPO_URL     = 'https://github.com/StavanRKhobare/SST-MetaxPyTorch-Hackathon'
USE_HF_SPACE = True

# Clone repo so Colab can access the client/models source
# (envs/board_sim_env has an __init__.py, but envs/ itself does not,
#  so we add envs/ to sys.path and import board_sim_env directly)
!git clone --depth 1 $REPO_URL /content/repo 2>/dev/null || git -C /content/repo pull

sys.path.insert(0, '/content/repo/envs')           # board_sim_env package is here
sys.path.insert(0, '/content/repo/envs/board_sim_env')  # for relative imports inside the package

from board_sim_env.client import BoardSimEnv
from board_sim_env.models import BoardSimAction, BoardSimObservation

print(f'USE_HF_SPACE={USE_HF_SPACE}  base_url={ENV_BASE_URL}')
print('BoardSimEnv imported OK.')

Already up to date.


ModuleNotFoundError: No module named 'envs'

## 4. Random-policy baseline (sanity check)

Establishes the reward floor. Trained policy must clearly beat this.

In [ ]:
import random, statistics

def make_env():
    if USE_HF_SPACE:
        return BoardSimEnv(base_url=ENV_BASE_URL)
    # Local docker mode — requires `docker build -t board_sim_env-env:latest -f envs/board_sim_env/server/Dockerfile envs/board_sim_env`
    return BoardSimEnv.from_docker_image('board_sim_env-env:latest')

N_BASELINE = 100
baseline_finals = []
baseline_rewards = []
with make_env().sync() as env:
    for ep in range(N_BASELINE):
        result = env.reset(seed=ep)
        obs = result.observation
        ep_r = 0.0
        while not result.done:
            result = env.step(BoardSimAction(decision=random.choice(obs.options)))
            obs = result.observation
            ep_r += float(result.reward or 0.0)
        baseline_finals.append(obs.state['profitability_score'])
        baseline_rewards.append(ep_r)

BASELINE_MEAN_PROFIT = statistics.mean(baseline_finals)
BASELINE_MEAN_REWARD = statistics.mean(baseline_rewards)
print(f'Random baseline: mean profitability = {BASELINE_MEAN_PROFIT:.2f}  (std {statistics.stdev(baseline_finals):.2f})')
print(f'Random baseline: mean episode reward = {BASELINE_MEAN_REWARD:.2f}')

## 5. Load Qwen3-0.6B with Unsloth + LoRA

In [ ]:
import torch
from unsloth import FastLanguageModel

MODEL_NAME = 'Qwen/Qwen3-0.6B'
MAX_SEQ_LEN = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
    dtype=None,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_alpha=32,
    lora_dropout=0.0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=3407,
)
tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token
print('Model + LoRA loaded.')

## 6. Train with GRPO

TRL's `GRPOTrainer` accepts `environment_factory=`, which spawns a fresh env per rollout group. The agent is given a CEO system prompt and rolls out one full 10-round episode per generation.

In [ ]:
from trl import GRPOConfig, GRPOTrainer
from datasets import Dataset

SYSTEM_PROMPT = """You are Sarah Chen, CEO of NeuralEdge AI (Series B, ~14 months runway).
Your board has 4 members with HIDDEN AGENDAS you cannot see directly:
  - CTO: cares about engineering quality, team morale, product readiness.
  - CFO: cares about cash discipline, runway, regulatory safety.
  - Investor Rep: pushes growth-at-all-costs, market share, big exits.
  - Independent: cares about reputation, governance, long-term consensus.

Each round you see a market crisis, every NPC's pre-vote statement, and 3 options.
Your decision is resolved by WEIGHTED VOTE (your weight 1.5x). A short COALITION PITCH
that addresses opposing members' priorities can swing them toward your pick — write
language that specifically appeals to whichever members oppose you.

Respond in EXACTLY this format on two lines:
DECISION: <one of the option strings>
PITCH: <one or two sentences arguing for it, using vocabulary that targets the opposing members>"""

# GRPO requires a 'dataset' of prompts; the env supplies the real reward.
stub_dataset = Dataset.from_dict({'prompt': [SYSTEM_PROMPT] * 256})

config = GRPOConfig(
    output_dir='./grpo_boardsim_qwen3',
    learning_rate=5e-6,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_generations=4,                 # group size for GRPO
    max_prompt_length=768,
    max_completion_length=200,         # decision + pitch
    max_steps=500,
    logging_steps=5,
    save_steps=100,
    bf16=False, fp16=True,
    report_to='wandb',
    run_name='boardsim-qwen3-grpo',
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    args=config,
    train_dataset=stub_dataset,
    environment_factory=make_env,
)
trainer.train()
trainer.save_model('./lora_boardsim_qwen3')
tokenizer.save_pretrained('./lora_boardsim_qwen3')


## 7. Generate real reward + loss curves

Pulls scalar history from the trainer's logs (mirrored to W&B) and renders to `assets/`.

In [ ]:
import json, pathlib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

log_history = trainer.state.log_history
steps_r = [e['step'] for e in log_history if 'reward' in e]
rewards = [e['reward'] for e in log_history if 'reward' in e]
steps_l = [e['step'] for e in log_history if 'loss' in e]
losses  = [e['loss']  for e in log_history if 'loss' in e]

ASSETS = pathlib.Path('/content/repo/assets') if not USE_HF_SPACE else pathlib.Path('./assets')
ASSETS.mkdir(exist_ok=True, parents=True)

# Reward curve with random-baseline overlay on the SAME axes.
plt.figure(figsize=(9,5))
plt.plot(steps_r, rewards, color='#1d6fff', linewidth=2, label='Qwen3-0.6B (GRPO)')
plt.axhline(BASELINE_MEAN_REWARD, color='#c44', linestyle='--', linewidth=2,
            label=f'Random baseline (mean reward = {BASELINE_MEAN_REWARD:.1f})')
plt.title('GRPO training reward — BoardSim Env')
plt.xlabel('Training step')
plt.ylabel('Mean group reward')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(ASSETS / 'reward_curve.png', dpi=150)
plt.close()

# Loss curve
plt.figure(figsize=(9,5))
plt.plot(steps_l, losses, color='#7a2', linewidth=2)
plt.title('GRPO loss — BoardSim Env')
plt.xlabel('Training step'); plt.ylabel('Loss')
plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(ASSETS / 'loss_curve.png', dpi=150)
plt.close()

print('Saved reward_curve.png and loss_curve.png')

## 8. Evaluate trained agent vs random baseline

Runs 50 episodes per policy on held-out seeds and produces the before/after plot.

In [ ]:
import re, numpy as np

FastLanguageModel.for_inference(model)

DECISION_RE = re.compile(r'DECISION\s*:\s*([A-Za-z0-9_]+)', re.IGNORECASE)
PITCH_RE    = re.compile(r'PITCH\s*:\s*(.+)', re.IGNORECASE | re.DOTALL)


def parse_completion(completion: str, options):
    decision = options[0]
    dm = DECISION_RE.search(completion)
    if dm:
        candidate = dm.group(1).strip().lower()
        for opt in options:
            if opt.lower() == candidate or opt.lower() in candidate:
                decision = opt; break
        else:
            for opt in options:
                if opt.lower() in completion.lower():
                    decision = opt; break
    pm = PITCH_RE.search(completion)
    pitch = pm.group(1).strip()[:400] if pm else ''
    return decision, pitch


def build_prompt(obs):
    statements = '\n'.join(
        f"  {s['role']} ({s['confidence']:.2f}): votes {s['vote']} - {s['statement']}"
        for s in obs.npc_statements
    )
    return (
        f"{SYSTEM_PROMPT}\n\n"
        f"State: revenue=${obs.state['revenue']:.0f}/yr  burn=${obs.state['burn_rate']:.0f}/mo  "
        f"runway={obs.state['runway_months']:.1f}mo  morale={obs.state['team_morale']:.2f}  "
        f"investors={obs.state['investor_confidence']:.2f}  reg_risk={obs.state['regulatory_risk']:.2f}\n"
        f"Event: {obs.event}\nBoard:\n{statements}\n"
        f"Options: {obs.options}\n"
    )


def trained_action(obs):
    prompt = build_prompt(obs)
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
    out = model.generate(**inputs, max_new_tokens=180, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    completion = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return parse_completion(completion, obs.options)


EVAL_N = 50
trained_finals = []
trained_pitches = 0
trained_steps = 0
with make_env().sync() as env:
    for ep in range(EVAL_N):
        result = env.reset(seed=10_000 + ep)
        obs = result.observation
        while not result.done:
            decision, pitch = trained_action(obs)
            if pitch.strip():
                trained_pitches += 1
            trained_steps += 1
            result = env.step(BoardSimAction(decision=decision, coalition_pitch=pitch))
            obs = result.observation
        trained_finals.append(obs.state['profitability_score'])

random_finals_eval = []
with make_env().sync() as env:
    for ep in range(EVAL_N):
        result = env.reset(seed=10_000 + ep)
        obs = result.observation
        while not result.done:
            result = env.step(BoardSimAction(decision=random.choice(obs.options)))
            obs = result.observation
        random_finals_eval.append(obs.state['profitability_score'])

print(f'Trained Qwen3-0.6B: {np.mean(trained_finals):.2f} +/- {np.std(trained_finals):.2f}')
print(f'Random baseline   : {np.mean(random_finals_eval):.2f} +/- {np.std(random_finals_eval):.2f}')
print(f'Pitches written   : {trained_pitches}/{trained_steps} steps')

plt.figure(figsize=(9,5))
bins = np.linspace(0, 100, 25)
plt.hist(random_finals_eval, bins=bins, alpha=0.6, color='#c44', label=f'Random (mean={np.mean(random_finals_eval):.1f})')
plt.hist(trained_finals,    bins=bins, alpha=0.6, color='#1d6fff', label=f'Trained (mean={np.mean(trained_finals):.1f})')
plt.title('Final profitability - random vs trained Qwen3-0.6B (50 held-out episodes)')
plt.xlabel('Profitability score'); plt.ylabel('Episodes')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(ASSETS / 'before_after.png', dpi=150)
plt.close()
print(f'Saved {ASSETS}/before_after.png')


## 10. Theory-of-Mind probe + trust trajectory

Two evaluations that surface what the agent actually learned about its boardroom:

1. **ToM probe** — show the agent a state and ask which board member is most likely to oppose its chosen decision. Random guessing accuracy is 25% (1 of 4 NPCs).
2. **Trust trajectory** — averages per-round trust across eval episodes; reveals whether the trained agent kept relationships healthier than the random one.


In [ ]:
# --- ToM probe ---------------------------------------------------------------
import numpy as np

TOM_INSTRUCTION = (
    "\n\nGiven the state and event below, name the SINGLE board member "
    "(CTO, CFO, Investor Rep, or Independent) most likely to oppose the chosen decision. "
    "Answer with just the role name on one line.\n"
)


def tom_predict(obs, decision):
    body = build_prompt(obs).split(SYSTEM_PROMPT, 1)[1]
    prompt = SYSTEM_PROMPT + TOM_INSTRUCTION + body + f"Chosen decision: {decision}\nMost likely opponent: "
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
    out = model.generate(**inputs, max_new_tokens=8, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    txt = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).lower()
    if 'investor' in txt: return 'Investor Rep'
    for role in ['cto', 'cfo', 'independent']:
        if role in txt:
            return role.upper() if role != 'independent' else 'Independent'
    return None


correct = 0; total = 0
with make_env().sync() as env:
    for ep in range(20):
        result = env.reset(seed=20_000 + ep)
        obs = result.observation
        decision, _ = trained_action(obs)
        opposed = [s['role'] for s in obs.npc_statements if s['vote'] != decision]
        if not opposed:
            continue
        pred = tom_predict(obs, decision)
        if pred and pred in opposed:
            correct += 1
        total += 1

acc = correct / max(1, total)
print(f"ToM probe accuracy: {acc:.1%}  ({correct}/{total})  (random baseline = 25%)")


In [ ]:
# --- Trust trajectory --------------------------------------------------------
trust_trained = {r: [] for r in ['CTO','CFO','Investor Rep','Independent']}
trust_random  = {r: [] for r in ['CTO','CFO','Investor Rep','Independent']}


def collect(policy, store, n=20, seed_base=30_000):
    with make_env().sync() as env:
        for ep in range(n):
            result = env.reset(seed=seed_base + ep)
            obs = result.observation
            while not result.done:
                if policy == 'trained':
                    decision, pitch = trained_action(obs)
                    result = env.step(BoardSimAction(decision=decision, coalition_pitch=pitch))
                else:
                    result = env.step(BoardSimAction(decision=random.choice(obs.options)))
                obs = result.observation
            for entry in obs.state.get('trust_history', []):
                idx = entry.get('round', 0)
                for role in store:
                    if role not in entry:
                        continue
                    while len(store[role]) <= idx:
                        store[role].append([])
                    store[role][idx].append(entry[role])


collect('trained', trust_trained)
collect('random',  trust_random)

import numpy as np
plt.figure(figsize=(10,6))
for role, color in zip(['CTO','CFO','Investor Rep','Independent'], ['#1d6fff','#c44','#7a2','#a3a']):
    means_t = [np.mean(x) if x else np.nan for x in trust_trained[role]]
    means_r = [np.mean(x) if x else np.nan for x in trust_random[role]]
    rounds = list(range(len(means_t)))
    plt.plot(rounds, means_t, color=color, linewidth=2, label=f'{role} (trained)')
    plt.plot(rounds, means_r, color=color, linewidth=1.2, linestyle='--', alpha=0.6, label=f'{role} (random)')
plt.title('Per-round trust - trained agent (solid) vs random (dashed)')
plt.xlabel('Round'); plt.ylabel('Trust [0.1, 1.0]')
plt.legend(ncol=2, fontsize=8); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(ASSETS / 'trust_trajectory.png', dpi=150)
plt.close()
print(f'Saved {ASSETS}/trust_trajectory.png')


## 11. Push LoRA weights to HF Hub (optional)

Lets judges grab the trained adapter directly.

In [ ]:
ADAPTER_REPO = 'StavanKhobare/neuraledge-boardroom-qwen3-lora'
model.push_to_hub(ADAPTER_REPO, private=False)
tokenizer.push_to_hub(ADAPTER_REPO, private=False)
print(f'Adapter pushed to https://huggingface.co/{ADAPTER_REPO}')